# Análise ad hoc — Mercado de Campinas (SP)

Análise descritiva e detalhada da cidade para subsidiar decisões de expansão da FGV.
Combina KPIs socioeconômicos, mercado de trabalho formal (RAIS), densidade empresarial e pipeline universitário.

**Janela temporal:** 2020–2023 (pós-pandemia, mais relevante para tendências atuais).

**Fontes:**
- IBGE (`dim_municipio`) — perfil socioeconômico
- RAIS (`fct_empregos`, `fct_estabelecimentos`) — vínculos formais e empresas
- INEP (`fct_mercado_superior`) — graduação

**Estrutura:**
- Bloco 0 — Setup
- Bloco 1 — Perfil socioeconômico
- Bloco 2 — Mercado de trabalho (visão geral)
- Bloco 3 — Setores em alta e em queda
- Bloco 4 — Cargos (CBO) — demanda e tendência
- Bloco 5 — Densidade empresarial
- Bloco 6 — Pipeline universitário
- Bloco 7 — Síntese: quadrante de oportunidades

> Pré-requisito: `gcloud auth application-default login` rodado na máquina.
> Dependências: `pip install google-cloud-bigquery pandas plotly`

## Bloco 0 — Setup e parâmetros

In [ ]:
from google.cloud import bigquery
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.options.display.float_format = '{:,.2f}'.format
pd.options.display.max_rows = 200
pd.options.display.max_columns = 50

PROJECT      = 'project-a8f8452a-3033-4dd8-99a'
ID_MUNICIPIO = '3509007'   # Campinas
SIGLA_UF     = 'SP'
NOME_CIDADE  = 'Campinas'

# datasets
DS_DIM   = f'`{PROJECT}.raw_dimensions`'
DS_FACT  = f'`{PROJECT}.raw_facts`'

client = bigquery.Client(project=PROJECT)

def q(sql: str) -> pd.DataFrame:
    return client.query(sql).to_dataframe()

print(f'Conectado ao projeto {PROJECT}')
print(f'Cidade alvo: {NOME_CIDADE}/{SIGLA_UF} (id_municipio={ID_MUNICIPIO})')

## Bloco 1 — Perfil socioeconômico

Comparativo Campinas vs média de SP vs média do Brasil para os principais indicadores demográficos e de renda.

In [ ]:
sql = f"""
WITH cidade AS (
  SELECT 'Campinas' AS escopo, idhm, renda_per_capita, indice_gini,
         populacao_total, populacao_urbana, taxa_superior_25_mais, prop_pobreza
  FROM {DS_DIM}.dim_municipio
  WHERE id_municipio = '{ID_MUNICIPIO}'
),
uf AS (
  SELECT 'Média SP' AS escopo,
         AVG(idhm) AS idhm, AVG(renda_per_capita) AS renda_per_capita,
         AVG(indice_gini) AS indice_gini, AVG(populacao_total) AS populacao_total,
         AVG(populacao_urbana) AS populacao_urbana,
         AVG(taxa_superior_25_mais) AS taxa_superior_25_mais,
         AVG(prop_pobreza) AS prop_pobreza
  FROM {DS_DIM}.dim_municipio WHERE sigla_uf = '{SIGLA_UF}'
),
br AS (
  SELECT 'Média BR' AS escopo,
         AVG(idhm) AS idhm, AVG(renda_per_capita) AS renda_per_capita,
         AVG(indice_gini) AS indice_gini, AVG(populacao_total) AS populacao_total,
         AVG(populacao_urbana) AS populacao_urbana,
         AVG(taxa_superior_25_mais) AS taxa_superior_25_mais,
         AVG(prop_pobreza) AS prop_pobreza
  FROM {DS_DIM}.dim_municipio
)
SELECT * FROM cidade
UNION ALL SELECT * FROM uf
UNION ALL SELECT * FROM br
"""
df_perfil = q(sql).set_index('escopo').T
df_perfil['Δ% vs SP'] = ((df_perfil['Campinas'] - df_perfil['Média SP']) / df_perfil['Média SP'].abs() * 100).round(1)
df_perfil['Δ% vs BR'] = ((df_perfil['Campinas'] - df_perfil['Média BR']) / df_perfil['Média BR'].abs() * 100).round(1)
df_perfil

In [ ]:
# Visualização — barras horizontais comparando Campinas vs SP vs BR
metricas_plot = ['idhm', 'renda_per_capita', 'indice_gini', 'taxa_superior_25_mais', 'prop_pobreza']
labels = {
    'idhm': 'IDHM',
    'renda_per_capita': 'Renda per capita (R$)',
    'indice_gini': 'Índice de Gini',
    'taxa_superior_25_mais': '% pop 25+ c/ superior',
    'prop_pobreza': '% em pobreza'
}

fig = make_subplots(rows=2, cols=3, subplot_titles=[labels[m] for m in metricas_plot])
for i, m in enumerate(metricas_plot):
    row, col = (i // 3) + 1, (i % 3) + 1
    fig.add_trace(
        go.Bar(x=['Campinas', 'Média SP', 'Média BR'],
               y=df_perfil.loc[m, ['Campinas', 'Média SP', 'Média BR']].values,
               marker_color=['#0a3b6e', '#7393b3', '#cccccc'],
               showlegend=False),
        row=row, col=col
    )
fig.update_layout(height=600, title_text='Campinas vs Média SP vs Média BR — perfil socioeconômico')
fig.show()

## Bloco 2 — Mercado de trabalho formal (visão geral)

Total de vínculos, salário médio ponderado e evolução 2020–2022 (RAIS).

In [ ]:
sql = f"""
SELECT
  ano,
  SUM(total_vinculos) AS total_vinculos,
  ROUND(SAFE_DIVIDE(SUM(total_vinculos * salario_medio_reais), SUM(total_vinculos)), 2) AS salario_medio_ponderado,
  COUNT(DISTINCT subsetor_ibge) AS qtd_subsetores,
  COUNT(DISTINCT cbo_2002)      AS qtd_cargos_distintos
FROM {DS_FACT}.fct_empregos
WHERE id_municipio = '{ID_MUNICIPIO}' AND ano BETWEEN 2020 AND 2022
GROUP BY ano ORDER BY ano
"""
df_mt = q(sql)
df_mt

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('Total de vínculos formais', 'Salário médio ponderado (R$)'))
fig.add_trace(go.Scatter(x=df_mt['ano'], y=df_mt['total_vinculos'], mode='lines+markers+text',
                          text=[f"{v:,.0f}" for v in df_mt['total_vinculos']], textposition='top center',
                          line=dict(color='#0a3b6e', width=3), marker=dict(size=10)), row=1, col=1)
fig.add_trace(go.Scatter(x=df_mt['ano'], y=df_mt['salario_medio_ponderado'], mode='lines+markers+text',
                          text=[f"R$ {v:,.0f}" for v in df_mt['salario_medio_ponderado']], textposition='top center',
                          line=dict(color='#c9a13d', width=3), marker=dict(size=10)), row=1, col=2)
fig.update_layout(height=400, showlegend=False, title_text=f'{NOME_CIDADE} — evolução do mercado formal (2020–2022)')
fig.show()

## Bloco 3 — Setores em alta e em queda

Top subsetores por volume de vínculos em 2022, com variação % vs 2020. Quem cresceu, quem caiu.

In [ ]:
sql = f"""
WITH base AS (
  SELECT ano, descricao_subsetor,
         SUM(total_vinculos) AS vinculos,
         ROUND(SAFE_DIVIDE(SUM(total_vinculos * salario_medio_reais), SUM(total_vinculos)), 2) AS salario_medio
  FROM {DS_FACT}.fct_empregos
  WHERE id_municipio = '{ID_MUNICIPIO}' AND ano IN (2020, 2021, 2022)
  GROUP BY ano, descricao_subsetor
)
SELECT
  descricao_subsetor,
  MAX(IF(ano=2020, vinculos, NULL))     AS vinculos_2020,
  MAX(IF(ano=2021, vinculos, NULL))     AS vinculos_2021,
  MAX(IF(ano=2022, vinculos, NULL))     AS vinculos_2022,
  MAX(IF(ano=2022, salario_medio, NULL)) AS salario_medio_2022,
  ROUND(SAFE_DIVIDE(
    MAX(IF(ano=2022, vinculos, NULL)) - MAX(IF(ano=2020, vinculos, NULL)),
    NULLIF(MAX(IF(ano=2020, vinculos, NULL)), 0)
  ) * 100, 1) AS variacao_pct_20_22
FROM base
GROUP BY descricao_subsetor
ORDER BY vinculos_2022 DESC
"""
df_setores = q(sql)
df_setores

In [ ]:
# Top 10 em alta (apenas com volume relevante)
alta = df_setores[df_setores['vinculos_2022'] >= 500].sort_values('variacao_pct_20_22', ascending=False).head(10)
queda = df_setores[df_setores['vinculos_2022'] >= 500].sort_values('variacao_pct_20_22', ascending=True).head(10)
print('▲ TOP 10 SETORES EM ALTA (vol ≥ 500 em 2022)')
display(alta[['descricao_subsetor', 'vinculos_2020', 'vinculos_2022', 'variacao_pct_20_22', 'salario_medio_2022']])
print('\n▼ TOP 10 SETORES EM QUEDA (vol ≥ 500 em 2022)')
display(queda[['descricao_subsetor', 'vinculos_2020', 'vinculos_2022', 'variacao_pct_20_22', 'salario_medio_2022']])

In [ ]:
# Gráfico de barras divergente — top 15 setores por volume, cor = variação
top15 = df_setores.head(15).copy()
top15 = top15.sort_values('vinculos_2022', ascending=True)
cores = ['#2ca02c' if v and v > 0 else '#d62728' for v in top15['variacao_pct_20_22']]
fig = go.Figure(go.Bar(
    y=top15['descricao_subsetor'], x=top15['vinculos_2022'],
    orientation='h', marker_color=cores,
    text=[f"{v:,.0f} ({d:+.1f}%)" if pd.notna(d) else f"{v:,.0f}"
          for v, d in zip(top15['vinculos_2022'], top15['variacao_pct_20_22'])],
    textposition='outside'
))
fig.update_layout(height=600, title='Top 15 subsetores em Campinas — volume 2022 e Δ% vs 2020 (verde=cresceu, vermelho=caiu)',
                  xaxis_title='Vínculos formais 2022', margin=dict(l=200))
fig.show()

## Bloco 4 — Cargos (CBO): demanda e tendência

Mais granular. Top cargos por volume, top em ascensão, top em queda. Salário médio por cargo.

In [ ]:
sql = f"""
WITH base AS (
  SELECT ano, cbo_2002, descricao_cargo,
         SUM(total_vinculos) AS vinculos,
         ROUND(SAFE_DIVIDE(SUM(total_vinculos * salario_medio_reais), SUM(total_vinculos)), 2) AS salario_medio
  FROM {DS_FACT}.fct_empregos
  WHERE id_municipio = '{ID_MUNICIPIO}' AND ano IN (2020, 2022) AND descricao_cargo IS NOT NULL
  GROUP BY ano, cbo_2002, descricao_cargo
)
SELECT
  cbo_2002, descricao_cargo,
  MAX(IF(ano=2020, vinculos, NULL))      AS vinculos_2020,
  MAX(IF(ano=2022, vinculos, NULL))      AS vinculos_2022,
  MAX(IF(ano=2022, salario_medio, NULL)) AS salario_medio_2022,
  ROUND(SAFE_DIVIDE(
    MAX(IF(ano=2022, vinculos, NULL)) - MAX(IF(ano=2020, vinculos, NULL)),
    NULLIF(MAX(IF(ano=2020, vinculos, NULL)), 0)
  ) * 100, 1) AS variacao_pct_20_22
FROM base
GROUP BY cbo_2002, descricao_cargo
ORDER BY vinculos_2022 DESC
"""
df_cargos = q(sql)
print(f'{len(df_cargos)} cargos distintos com vínculos em Campinas')
df_cargos.head(30)

In [ ]:
# Top 20 em ascensão (volume mínimo 100 em 2022 para evitar ruído)
cargos_relevantes = df_cargos[df_cargos['vinculos_2022'] >= 100].copy()
ascensao = cargos_relevantes.sort_values('variacao_pct_20_22', ascending=False).head(20)
queda    = cargos_relevantes.sort_values('variacao_pct_20_22', ascending=True).head(20)

print('▲ TOP 20 CARGOS EM ASCENSÃO (vol ≥ 100 em 2022)')
display(ascensao[['descricao_cargo', 'vinculos_2020', 'vinculos_2022', 'variacao_pct_20_22', 'salario_medio_2022']])
print('\n▼ TOP 20 CARGOS EM QUEDA')
display(queda[['descricao_cargo', 'vinculos_2020', 'vinculos_2022', 'variacao_pct_20_22', 'salario_medio_2022']])

In [ ]:
# Scatter: volume × variação — quadrante de cargos
scatter_df = cargos_relevantes.dropna(subset=['variacao_pct_20_22'])
scatter_df = scatter_df[scatter_df['variacao_pct_20_22'].between(-100, 500)]  # remove outliers extremos
fig = px.scatter(
    scatter_df, x='vinculos_2022', y='variacao_pct_20_22',
    size='salario_medio_2022', hover_name='descricao_cargo',
    hover_data={'vinculos_2020': True, 'salario_medio_2022': ':.2f'},
    labels={'vinculos_2022': 'Vínculos 2022', 'variacao_pct_20_22': 'Variação % 2020→2022'},
    title='Cargos em Campinas — volume × crescimento (tamanho = salário médio)',
    log_x=True
)
fig.add_hline(y=0, line_dash='dash', line_color='gray')
fig.update_layout(height=600)
fig.show()

## Bloco 5 — Densidade empresarial

Estabelecimentos formais por ano, top setores e funcionários por estabelecimento (proxy de porte).

In [ ]:
sql = f"""
SELECT
  ano,
  SUM(total_estabelecimentos) AS estabelecimentos,
  SUM(total_vinculos_ativos)  AS vinculos_ativos,
  ROUND(SAFE_DIVIDE(SUM(total_vinculos_ativos), SUM(total_estabelecimentos)), 2) AS func_por_estab
FROM {DS_FACT}.fct_estabelecimentos
WHERE id_municipio = '{ID_MUNICIPIO}' AND ano BETWEEN 2020 AND 2023
GROUP BY ano ORDER BY ano
"""
df_estab = q(sql)
df_estab

In [ ]:
fig = make_subplots(rows=1, cols=3, subplot_titles=('Estabelecimentos', 'Vínculos ativos', 'Funcionários por estab'))
fig.add_trace(go.Bar(x=df_estab['ano'], y=df_estab['estabelecimentos'],
                     text=[f"{v:,.0f}" for v in df_estab['estabelecimentos']],
                     marker_color='#0a3b6e'), row=1, col=1)
fig.add_trace(go.Bar(x=df_estab['ano'], y=df_estab['vinculos_ativos'],
                     text=[f"{v:,.0f}" for v in df_estab['vinculos_ativos']],
                     marker_color='#7393b3'), row=1, col=2)
fig.add_trace(go.Bar(x=df_estab['ano'], y=df_estab['func_por_estab'],
                     text=[f"{v:.1f}" for v in df_estab['func_por_estab']],
                     marker_color='#c9a13d'), row=1, col=3)
fig.update_layout(height=400, showlegend=False, title='Densidade empresarial em Campinas (2020-2023)')
fig.show()

In [ ]:
# Top setores por estabelecimentos em 2023
sql = f"""
WITH base AS (
  SELECT ano, descricao_subsetor,
         SUM(total_estabelecimentos) AS estabs,
         SUM(total_vinculos_ativos)  AS vinculos
  FROM {DS_FACT}.fct_estabelecimentos
  WHERE id_municipio = '{ID_MUNICIPIO}' AND ano IN (2020, 2023)
  GROUP BY ano, descricao_subsetor
)
SELECT
  descricao_subsetor,
  MAX(IF(ano=2020, estabs, NULL))   AS estabs_2020,
  MAX(IF(ano=2023, estabs, NULL))   AS estabs_2023,
  MAX(IF(ano=2023, vinculos, NULL)) AS vinculos_2023,
  ROUND(SAFE_DIVIDE(MAX(IF(ano=2023, vinculos, NULL)), NULLIF(MAX(IF(ano=2023, estabs, NULL)), 0)), 1) AS func_por_estab_2023,
  ROUND(SAFE_DIVIDE(
    MAX(IF(ano=2023, estabs, NULL)) - MAX(IF(ano=2020, estabs, NULL)),
    NULLIF(MAX(IF(ano=2020, estabs, NULL)), 0)
  ) * 100, 1) AS variacao_pct_estab_20_23
FROM base
GROUP BY descricao_subsetor
ORDER BY estabs_2023 DESC
"""
df_estab_setor = q(sql)
df_estab_setor

## Bloco 6 — Pipeline universitário

Quem está se formando em Campinas — input para programas de pós-graduação e MBA.

In [ ]:
# Visão geral por ano
sql = f"""
SELECT
  ano,
  SUM(total_vagas)        AS vagas,
  SUM(total_inscritos)    AS inscritos,
  SUM(total_ingressantes) AS ingressantes,
  SUM(total_matriculas)   AS matriculas,
  SUM(total_concluintes)  AS concluintes,
  COUNT(DISTINCT id_ies)  AS qtd_ies,
  COUNT(DISTINCT id_curso) AS qtd_cursos
FROM {DS_FACT}.fct_mercado_superior
WHERE id_municipio = '{ID_MUNICIPIO}' AND ano BETWEEN 2020 AND 2023
GROUP BY ano ORDER BY ano
"""
df_sup = q(sql)
df_sup

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=('Matrículas e concluintes', 'Quantidade de cursos e IES'))
fig.add_trace(go.Scatter(x=df_sup['ano'], y=df_sup['matriculas'], mode='lines+markers', name='Matrículas',
                          line=dict(color='#0a3b6e', width=3)), row=1, col=1)
fig.add_trace(go.Scatter(x=df_sup['ano'], y=df_sup['concluintes'], mode='lines+markers', name='Concluintes',
                          line=dict(color='#c9a13d', width=3)), row=1, col=1)
fig.add_trace(go.Bar(x=df_sup['ano'], y=df_sup['qtd_cursos'], name='Cursos',
                     marker_color='#7393b3'), row=1, col=2)
fig.add_trace(go.Bar(x=df_sup['ano'], y=df_sup['qtd_ies'], name='IES',
                     marker_color='#0a3b6e'), row=1, col=2)
fig.update_layout(height=400, title='Ensino superior em Campinas — evolução (2020-2023)')
fig.show()

In [ ]:
# Top áreas — concluintes 2022 e variação 2020→2022
sql = f"""
WITH base AS (
  SELECT ano, nome_area_geral, nome_area_especifica,
         SUM(total_concluintes) AS concluintes,
         SUM(total_matriculas)  AS matriculas
  FROM {DS_FACT}.fct_mercado_superior
  WHERE id_municipio = '{ID_MUNICIPIO}' AND ano IN (2020, 2022)
  GROUP BY ano, nome_area_geral, nome_area_especifica
)
SELECT
  nome_area_geral, nome_area_especifica,
  MAX(IF(ano=2020, concluintes, NULL)) AS concluintes_2020,
  MAX(IF(ano=2022, concluintes, NULL)) AS concluintes_2022,
  MAX(IF(ano=2022, matriculas, NULL))  AS matriculas_2022,
  ROUND(SAFE_DIVIDE(
    MAX(IF(ano=2022, concluintes, NULL)) - MAX(IF(ano=2020, concluintes, NULL)),
    NULLIF(MAX(IF(ano=2020, concluintes, NULL)), 0)
  ) * 100, 1) AS variacao_pct_20_22
FROM base
GROUP BY nome_area_geral, nome_area_especifica
ORDER BY concluintes_2022 DESC
"""
df_areas = q(sql)
df_areas

In [ ]:
# Áreas em alta vs queda (vol mín 50 concluintes em 2022)
areas_rel = df_areas[df_areas['concluintes_2022'] >= 50].copy()
areas_alta  = areas_rel.sort_values('variacao_pct_20_22', ascending=False).head(15)
areas_queda = areas_rel.sort_values('variacao_pct_20_22', ascending=True).head(15)
print('▲ TOP 15 ÁREAS DE FORMAÇÃO EM ALTA (concl ≥ 50 em 2022)')
display(areas_alta[['nome_area_geral', 'nome_area_especifica', 'concluintes_2020', 'concluintes_2022', 'variacao_pct_20_22']])
print('\n▼ TOP 15 ÁREAS EM QUEDA')
display(areas_queda[['nome_area_geral', 'nome_area_especifica', 'concluintes_2020', 'concluintes_2022', 'variacao_pct_20_22']])

In [ ]:
# Pública vs privada, EAD vs presencial
sql = f"""
SELECT
  ano, rede, modalidade_ensino,
  SUM(total_matriculas) AS matriculas,
  SUM(total_concluintes) AS concluintes
FROM {DS_FACT}.fct_mercado_superior
WHERE id_municipio = '{ID_MUNICIPIO}' AND ano BETWEEN 2020 AND 2022
GROUP BY ano, rede, modalidade_ensino
ORDER BY ano, rede, modalidade_ensino
"""
df_perfil_ies = q(sql)
df_perfil_ies

In [ ]:
fig = px.bar(df_perfil_ies, x='ano', y='matriculas', color='rede', barmode='group',
             facet_col='modalidade_ensino',
             title='Matrículas em Campinas — rede × modalidade × ano',
             color_discrete_map={'Pública': '#0a3b6e', 'Privada': '#c9a13d'})
fig.update_layout(height=400)
fig.show()

In [ ]:
# Top IES por concluintes (último ano disponível)
sql = f"""
WITH ano_max AS (SELECT MAX(ano) AS ano FROM {DS_FACT}.fct_mercado_superior WHERE id_municipio = '{ID_MUNICIPIO}')
SELECT
  m.nome_ies, m.rede, m.modalidade_ensino,
  SUM(m.total_matriculas)  AS matriculas,
  SUM(m.total_concluintes) AS concluintes,
  COUNT(DISTINCT m.id_curso) AS cursos_ofertados
FROM {DS_FACT}.fct_mercado_superior m
JOIN ano_max a ON m.ano = a.ano
WHERE m.id_municipio = '{ID_MUNICIPIO}'
GROUP BY m.nome_ies, m.rede, m.modalidade_ensino
ORDER BY concluintes DESC
"""
df_ies = q(sql)
df_ies

## Bloco 7 — Síntese: quadrante de oportunidades

Cruza setores em crescimento × volume formal × salário médio para identificar onde está o público-alvo de mais alto potencial.

In [ ]:
# Quadrante de setores: volume (x) × crescimento (y), tamanho = salário
quad = df_setores.dropna(subset=['variacao_pct_20_22']).copy()
quad = quad[quad['vinculos_2022'] >= 200]
quad = quad[quad['variacao_pct_20_22'].between(-80, 200)]
fig = px.scatter(
    quad, x='vinculos_2022', y='variacao_pct_20_22',
    size='salario_medio_2022', hover_name='descricao_subsetor',
    hover_data={'vinculos_2020': True, 'salario_medio_2022': ':.2f'},
    text='descricao_subsetor',
    labels={'vinculos_2022': 'Vínculos 2022', 'variacao_pct_20_22': 'Crescimento % 2020→2022'},
    title='Quadrante de oportunidades — Campinas (tamanho = salário médio)',
    log_x=True
)
fig.add_hline(y=0, line_dash='dash', line_color='gray')
fig.update_traces(textposition='top center', textfont_size=9)
fig.update_layout(height=700)
fig.show()

In [ ]:
# Tabela síntese — top 10 setores priorizados (volume + crescimento + salário)
sintese = df_setores[df_setores['vinculos_2022'].notna() & df_setores['variacao_pct_20_22'].notna()].copy()
sintese = sintese[sintese['vinculos_2022'] >= 500]
sintese['rank_volume'] = sintese['vinculos_2022'].rank(ascending=False)
sintese['rank_crescimento'] = sintese['variacao_pct_20_22'].rank(ascending=False)
sintese['rank_salario'] = sintese['salario_medio_2022'].rank(ascending=False)
sintese['score_combinado'] = (sintese['rank_volume'] + sintese['rank_crescimento'] + sintese['rank_salario']) / 3
sintese = sintese.sort_values('score_combinado').head(10)
print('TOP 10 SETORES PRIORIZADOS (média dos rankings de volume, crescimento e salário — menor = melhor)')
sintese[['descricao_subsetor', 'vinculos_2022', 'variacao_pct_20_22', 'salario_medio_2022', 'score_combinado']]

---
## Próximos passos

1. Cruzar os top setores priorizados com tendências empresariais externas (relatórios setoriais, notícias).
2. Validar áreas de formação em alta com portfólio FGV (graduação → pós).
3. Identificar empresas-alvo dos setores priorizados via `fct_empresas` (CNPJ + capital social + porte).
4. Comparar Campinas com outras praças foco — usar a página de comparativo do dashboard.